In [1]:
import os

project_path = r"C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS"

os.chdir(project_path)

print("Project folder:", os.getcwd())
print("Folders:", os.listdir())

Project folder: C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS
Folders: ['.ipynb_checkpoints', 'app.py', 'Data', 'Intelligent_ATS.ipynb', 'models', 'Output.pdf', 'README.md.txt', 'Report', 'Requirements.txt.txt', 'Result.pdf', 'Sample Resume']


In [2]:
import pandas as pd
import numpy as np
import re
import os

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

try:
    import fitz
    print("PDF library: Available")
except ImportError:
    print("PDF library: Not installed")

try:
    from docx import Document
    print("DOCX library: Available")
except ImportError:
    print("DOCX library: Not installed")

try:
    from sentence_transformers import SentenceTransformer
    print("Sentence Transformers: Available")
except ImportError:
    print("Sentence Transformers: Not installed")

print("Basic library check completed!")

Pandas: 2.3.3
NumPy: 2.2.6
PDF library: Available
DOCX library: Available
Sentence Transformers: Available
Basic library check completed!


In [3]:
try:
    from sentence_transformers import SentenceTransformer
    print("Sentence Transformers: Available")
except ImportError as e:
    print("Not installed:", e)

Sentence Transformers: Available


In [4]:
import pandas as pd

resumes = [
    {
        "Candidate": "Candidate_A",
        "Resume_Text": """
        BTech Computer Science graduate.
        Skills: Python, SQL, Pandas, NumPy, Excel, Power BI.
        Experience in data cleaning, data analysis and dashboards.
        Project: Sales Data Analysis using Python and Power BI.
        """
    },
    {
        "Candidate": "Candidate_B",
        "Resume_Text": """
        Computer Science graduate with knowledge of Java, HTML,
        CSS and JavaScript. Built a website using MySQL.
        Interested in web development.
        """
    },
    {
        "Candidate": "Candidate_C",
        "Resume_Text": """
        Data Analyst with skills in Python, SQL, Excel, Tableau,
        statistics and machine learning.
        Project: Customer Data Analysis and visualization.
        """
    }
]

resume_df = pd.DataFrame(resumes)

display(resume_df)

,Candidate,Resume_Text
0,Candidate_A,\n BTech Computer Science graduate.\n ...
1,Candidate_B,\n Computer Science graduate with knowl...
2,Candidate_C,"\n Data Analyst with skills in Python, ..."


In [5]:
job_description = """
We are looking for a Data Analyst.

Required skills:
Python, SQL, Excel, Power BI, Tableau, Pandas, NumPy,
Statistics, Data Cleaning, Data Analysis and Data Visualization.

The candidate should be able to analyze datasets,
create dashboards, generate reports and communicate
data-driven insights.

Education: BTech or Bachelor's degree in Computer Science,
Information Technology, Statistics or a related field.
"""

print("JOB DESCRIPTION")
print("-" * 50)
print(job_description)

JOB DESCRIPTION
--------------------------------------------------

We are looking for a Data Analyst.

Required skills:
Python, SQL, Excel, Power BI, Tableau, Pandas, NumPy,
Statistics, Data Cleaning, Data Analysis and Data Visualization.

The candidate should be able to analyze datasets,
create dashboards, generate reports and communicate
data-driven insights.

Education: BTech or Bachelor's degree in Computer Science,
Information Technology, Statistics or a related field.



In [6]:
import re

skills_list = [
    "Python", "SQL", "Excel", "Power BI", "Tableau",
    "Pandas", "NumPy", "Statistics", "Data Cleaning",
    "Data Analysis", "Data Visualization",
    "Machine Learning", "Deep Learning",
    "MySQL", "Communication"
]

def extract_skills(text):
    text = text.lower()
    found_skills = []

    for skill in skills_list:
        pattern = r"\b" + re.escape(skill.lower()) + r"\b"

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills


resume_df["Extracted_Skills"] = resume_df["Resume_Text"].apply(extract_skills)

jd_skills = extract_skills(job_description)

print("Skills required for the job:")
print(jd_skills)

print("\nSkills extracted from each resume:")
display(resume_df[["Candidate", "Extracted_Skills"]])

Skills required for the job:
['Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Pandas', 'NumPy', 'Statistics', 'Data Cleaning', 'Data Analysis', 'Data Visualization']

Skills extracted from each resume:


,Candidate,Extracted_Skills
0,Candidate_A,"[Python, SQL, Excel, Power BI, Pandas, NumPy, ..."
1,Candidate_B,[MySQL]
2,Candidate_C,"[Python, SQL, Excel, Tableau, Statistics, Data..."


In [7]:
def calculate_skill_match(candidate_skills, required_skills):
    matched = [
        skill for skill in required_skills
        if skill.lower() in [s.lower() for s in candidate_skills]
    ]

    missing = [
        skill for skill in required_skills
        if skill.lower() not in [s.lower() for s in candidate_skills]
    ]

    score = (len(matched) / len(required_skills)) * 100

    return pd.Series({
        "Matched_Skills": matched,
        "Missing_Skills": missing,
        "Skill_Match_Score": round(score, 2)
    })


match_results = resume_df["Extracted_Skills"].apply(
    lambda skills: calculate_skill_match(skills, jd_skills)
)

resume_df = pd.concat([resume_df, match_results], axis=1)

display(
    resume_df[
        ["Candidate", "Matched_Skills", "Missing_Skills", "Skill_Match_Score"]
    ]
)

,Candidate,Matched_Skills,Missing_Skills,Skill_Match_Score
0,Candidate_A,"[Python, SQL, Excel, Power BI, Pandas, NumPy, ...","[Tableau, Statistics, Data Visualization]",72.73
1,Candidate_B,[],"[Python, SQL, Excel, Power BI, Tableau, Pandas...",0.00
2,Candidate_C,"[Python, SQL, Excel, Tableau, Statistics, Data...","[Power BI, Pandas, NumPy, Data Cleaning, Data ...",54.55


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("AI model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

AI model loaded successfully!


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

# Convert Job Description into AI embeddings
jd_embedding = model.encode([job_description])

# Convert each resume into AI embeddings
resume_embeddings = model.encode(
    resume_df["Resume_Text"].tolist()
)

# Calculate semantic similarity
similarity_scores = cosine_similarity(
    resume_embeddings,
    jd_embedding
).flatten()

# Convert to percentage scale
resume_df["Semantic_Similarity_Score"] = (
    similarity_scores * 100
).round(2)

display(
    resume_df[
        ["Candidate", "Skill_Match_Score", "Semantic_Similarity_Score"]
    ]
)

,Candidate,Skill_Match_Score,Semantic_Similarity_Score
0,Candidate_A,72.73,70.110001
1,Candidate_B,0.00,32.740002
2,Candidate_C,54.55,80.209999


In [10]:
# Combine both scores: 50% skill match + 50% semantic similarity

resume_df["Final_Score"] = (
    0.5 * resume_df["Skill_Match_Score"]
    + 0.5 * resume_df["Semantic_Similarity_Score"]
).round(2)

# Rank candidates from highest to lowest score
resume_df["Rank"] = (
    resume_df["Final_Score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Sort candidates by rank
ranked_candidates = resume_df.sort_values(
    by="Final_Score",
    ascending=False
).reset_index(drop=True)

display(
    ranked_candidates[
        [
            "Rank",
            "Candidate",
            "Skill_Match_Score",
            "Semantic_Similarity_Score",
            "Final_Score"
        ]
    ]
)

,Rank,Candidate,Skill_Match_Score,Semantic_Similarity_Score,Final_Score
0,1,Candidate_A,72.73,70.110001,71.42
1,2,Candidate_C,54.55,80.209999,67.38
2,3,Candidate_B,0.00,32.740002,16.37


In [11]:
import os

resume_folder = os.path.join(os.getcwd(), "Sample Resume")

files = [
    file for file in os.listdir(resume_folder)
    if file.lower().endswith((".pdf", ".docx"))
]

print("Total resumes found:", len(files))
print("Resume files:")

for file in files:
    print(file)

Total resumes found: 6
Resume files:
apurva cv new.docx
Nikita Salunke (3)-1.pdf
Vaishnavi Dhumal AI Resume.docx
Vaishnavi Dhumal DA Resume..pdf
Vaishnavi Dhumal MIS Resume.docx
webResume.pdf


In [12]:
import os
import pymupdf
from docx import Document

resume_folder = os.path.join(os.getcwd(), "Sample Resume")

def extract_resume_text(file_path):
    if file_path.lower().endswith(".pdf"):
        text = ""
        with pymupdf.open(file_path) as pdf:
            for page in pdf:
                text += page.get_text() + "\n"
        return text

    elif file_path.lower().endswith(".docx"):
        doc = Document(file_path)
        text = "\n".join(
            paragraph.text for paragraph in doc.paragraphs
        )

        # Extract text from tables, if any
        for table in doc.tables:
            for row in table.rows:
                text += "\n" + " ".join(
                    cell.text for cell in row.cells
                )

        return text

    return ""

resume_data = []

for filename in files:
    file_path = os.path.join(resume_folder, filename)
    text = extract_resume_text(file_path)

    resume_data.append({
        "Candidate": filename,
        "Resume_Text": text
    })

actual_resume_df = pd.DataFrame(resume_data)

print("Total resumes processed:", len(actual_resume_df))

for _, row in actual_resume_df.iterrows():
    print("\nFile:", row["Candidate"])
    print("Extracted characters:", len(row["Resume_Text"]))
    print("Preview:", row["Resume_Text"][:250].replace("\n", " "))

Total resumes processed: 6

File: apurva cv new.docx
Extracted characters: 1066
Preview: Apurva Sharad Mandhare Pune, Maharashtra|+91 8698241946| apurvamandhre2704@gmail.com | https://www.linkedin.com/in/apurva-mandhare-65a85332b   PROFESSIONAL SUMMARY  A student currently pursuing a B.com from Shree Siddhivinayak mahila mahavidyalay . A

File: Nikita Salunke (3)-1.pdf
Extracted characters: 2739
Preview: Nikita Salunke  Pune, India | nikitasalunke2209@gmail.com | +91 8010265981 |  Linkedin.com/in/Nikita-salunke |   GitHub: nikita220908    Professional Summary  Java Developer (2026 Graduate) with hands-on experience building scalable web applications 

File: Vaishnavi Dhumal AI Resume.docx
Extracted characters: 2866
Preview: VAISHNAVI DHUMAL Computer Science & Engineering Graduate | Python | AI/ML | Data Analytics  Pune, India | dhumalvaishnavi003@gmail.com | +91-9284173533 |    LinkedIn: https://www.linkedin.com/in/vaishnavi-dhumal26    GitHub: https://github.com/Vaishn

File: Vaishnav

In [13]:
skills_list = [
    "Python", "SQL", "Excel", "Power BI", "Tableau",
    "Pandas", "NumPy", "Statistics", "Data Cleaning",
    "Data Analysis", "Data Visualization",
    "Machine Learning", "Deep Learning",
    "MySQL", "Communication", "Artificial Intelligence",
    "Generative AI", "NLP", "Flask", "Streamlit",
    "Java", "HTML", "CSS", "JavaScript", "Bootstrap",
    "Spring Boot", "REST API", "Microsoft Office",
    "VLOOKUP", "XLOOKUP", "Dashboard", "Reporting",
    "React", "Node.js", "MongoDB", "C++", "C",
    "PowerPoint", "Word", "Data Entry", "Accounting",
    "Tally", "UI/UX", "Web Development"
]

actual_resume_df["Extracted_Skills"] = (
    actual_resume_df["Resume_Text"].apply(extract_skills)
)

jd_skills = extract_skills(job_description)

print("Required Data Analyst skills:")
print(jd_skills)

print("\nSkills found in each resume:")

for _, row in actual_resume_df.iterrows():
    print("\nResume:", row["Candidate"])
    print("Skills:", row["Extracted_Skills"])

Required Data Analyst skills:
['Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Pandas', 'NumPy', 'Statistics', 'Data Cleaning', 'Data Analysis', 'Data Visualization']

Skills found in each resume:

Resume: apurva cv new.docx
Skills: ['Power BI', 'Tally']

Resume: Nikita Salunke (3)-1.pdf
Skills: ['SQL', 'MySQL', 'Java', 'HTML', 'CSS', 'JavaScript', 'Bootstrap', 'Spring Boot', 'React', 'C']

Resume: Vaishnavi Dhumal AI Resume.docx
Skills: ['Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Pandas', 'NumPy', 'Data Cleaning', 'Data Analysis', 'Data Visualization', 'Machine Learning', 'MySQL', 'Generative AI', 'Dashboard']

Resume: Vaishnavi Dhumal DA Resume..pdf
Skills: ['Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Pandas', 'NumPy', 'Statistics', 'Data Cleaning', 'Data Analysis', 'Data Visualization', 'MySQL', 'Dashboard']

Resume: Vaishnavi Dhumal MIS Resume.docx
Skills: ['Python', 'SQL', 'Excel', 'Power BI', 'Tableau', 'Pandas', 'NumPy', 'Statistics', 'Data Cleaning', 'Data Analys

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate skill match scores
match_results = actual_resume_df["Extracted_Skills"].apply(
    lambda skills: calculate_skill_match(skills, jd_skills)
)

# Remove old score columns if this cell is run again
score_columns = [
    "Matched_Skills",
    "Missing_Skills",
    "Skill_Match_Score",
    "Semantic_Similarity_Score",
    "Final_Score",
    "Rank"
]

actual_resume_df = actual_resume_df.drop(
    columns=[c for c in score_columns if c in actual_resume_df.columns]
)

actual_resume_df = pd.concat(
    [actual_resume_df, match_results], axis=1
)

# Calculate semantic similarity
jd_embedding = model.encode([job_description])
resume_embeddings = model.encode(
    actual_resume_df["Resume_Text"].tolist()
)

similarities = cosine_similarity(
    resume_embeddings, jd_embedding
).flatten()

actual_resume_df["Semantic_Similarity_Score"] = (
    similarities * 100
).round(2)

# Combine scores
actual_resume_df["Final_Score"] = (
    0.5 * actual_resume_df["Skill_Match_Score"]
    + 0.5 * actual_resume_df["Semantic_Similarity_Score"]
).round(2)

# Rank candidates
ranked_candidates = (
    actual_resume_df
    .sort_values("Final_Score", ascending=False)
    .reset_index(drop=True)
)

ranked_candidates["Rank"] = range(1, len(ranked_candidates) + 1)

display(
    ranked_candidates[
        [
            "Rank",
            "Candidate",
            "Skill_Match_Score",
            "Semantic_Similarity_Score",
            "Final_Score"
        ]
    ]
)

,Rank,Candidate,Skill_Match_Score,Semantic_Similarity_Score,Final_Score
0,1,Vaishnavi Dhumal DA Resume..pdf,100.00,64.050003,82.03
1,2,Vaishnavi Dhumal MIS Resume.docx,100.00,58.130001,79.07
2,3,Vaishnavi Dhumal AI Resume.docx,90.91,51.320000,71.11
3,4,apurva cv new.docx,9.09,27.120001,18.11
4,5,Nikita Salunke (3)-1.pdf,9.09,25.080000,17.08
5,6,webResume.pdf,0.00,26.709999,13.35


In [15]:
import os

report_folder = os.path.join(os.getcwd(), "Report")
os.makedirs(report_folder, exist_ok=True)

report_path = os.path.join(
    report_folder,
    "ATS_Candidate_Ranking.csv"
)

report_columns = [
    "Rank",
    "Candidate",
    "Skill_Match_Score",
    "Semantic_Similarity_Score",
    "Final_Score",
    "Matched_Skills",
    "Missing_Skills"
]

report_df = ranked_candidates[report_columns].copy()

report_df["Matched_Skills"] = report_df["Matched_Skills"].apply(
    lambda skills: ", ".join(skills)
)

report_df["Missing_Skills"] = report_df["Missing_Skills"].apply(
    lambda skills: ", ".join(skills)
)

report_df.to_csv(report_path, index=False, encoding="utf-8-sig")

print("Report saved successfully!")
print("File path:", report_path)

Report saved successfully!
File path: C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS\Report\ATS_Candidate_Ranking.csv


In [16]:
%pip show streamlit

Name: streamlit
Version: 1.65.0
Summary: A faster way to build and share data apps
Home-page: https://streamlit.io
Author: 
Author-email: Snowflake Inc <hello@streamlit.io>
License-Expression: Apache-2.0
Location: c:\users\dhuma\appdata\local\programs\python\python310\lib\site-packages
Requires: altair, anyio, click, httptools, itsdangerous, numpy, packaging, pandas, pillow, protobuf, pyarrow, pydeck, python-multipart, requests, starlette, toml, typing-extensions, uvicorn, watchdog, websockets
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [17]:
import os

print("Current Path:", os.getcwd())

Current Path: C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS


In [18]:
from sentence_transformers import SentenceTransformer

model_name = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Model loaded successfully!")
print("Model name:", model_name)
print("Model location:", model._modules[next(iter(model._modules))].auto_model.config._name_or_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!
Model name: sentence-transformers/all-MiniLM-L6-v2
Model location: sentence-transformers/all-MiniLM-L6-v2


In [19]:
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    "sentence-transformers/all-MiniLM-L6-v2",
    local_files_only=True
)

print("Actual Model Folder:", model_path)

Actual Model Folder: C:\Users\dhuma\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2\snapshots\1110a243fdf4706b3f48f1d95db1a4f5529b4d41


In [20]:
from sentence_transformers import SentenceTransformer
import os

project_path = r"C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS"
model_folder = os.path.join(project_path, "models", "all-MiniLM-L6-v2")

os.makedirs(model_folder, exist_ok=True)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model.save(model_folder)

print("Model saved at:", model_folder)
print("Folder exists:", os.path.exists(model_folder))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved at: C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS\models\all-MiniLM-L6-v2
Folder exists: True


In [21]:

import os
import re
import pandas as pd

resume_folder = r"C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS\Sample Resume"

resume_data = []

for filename in os.listdir(resume_folder):
    if filename.lower().endswith((".pdf", ".docx")):
        filepath = os.path.join(resume_folder, filename)

        resume_data.append({
            "Filename": filename,
            "File Size (KB)": round(os.path.getsize(filepath) / 1024, 2)
        })

resume_df = pd.DataFrame(resume_data)

print("Total resumes:", len(resume_df))
display(resume_df)


Total resumes: 6


,Filename,File Size (KB)
0,apurva cv new.docx,33.49
1,Nikita Salunke (3)-1.pdf,334.83
2,Vaishnavi Dhumal AI Resume.docx,28.20
3,Vaishnavi Dhumal DA Resume..pdf,208.69
4,Vaishnavi Dhumal MIS Resume.docx,23.85
5,webResume.pdf,112.67


In [22]:

import os
import re
import hashlib
import pandas as pd
import pymupdf
from docx import Document

resume_folder = r"C:\Users\dhuma\Desktop\Data Science\Deep Learning\Intelligent_ATS\Sample Resume"

def get_resume_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".pdf":
        with pymupdf.open(filepath) as pdf:
            return "\n".join(page.get_text() for page in pdf)

    elif ext == ".docx":
        doc = Document(filepath)
        parts = [p.text for p in doc.paragraphs]

        for table in doc.tables:
            for row in table.rows:
                parts.append(" ".join(cell.text for cell in row.cells))

        return "\n".join(parts)

    return ""

results = []

for filename in os.listdir(resume_folder):
    filepath = os.path.join(resume_folder, filename)

    if not filename.lower().endswith((".pdf", ".docx")):
        continue

    try:
        text = get_resume_text(filepath)
        normalized_text = re.sub(r"\s+", " ", text.lower()).strip()

        results.append({
            "Filename": filename,
            "Word Count": len(normalized_text.split()),
            "Text Length": len(normalized_text),
            "Empty Text": len(normalized_text) == 0,
            "Content Hash": hashlib.md5(
                normalized_text.encode("utf-8")
            ).hexdigest() if normalized_text else None
        })

    except Exception as e:
        results.append({
            "Filename": filename,
            "Word Count": 0,
            "Text Length": 0,
            "Empty Text": True,
            "Content Hash": None
        })
        print(f"Error reading {filename}: {e}")

quality_df = pd.DataFrame(results)

if not quality_df.empty:
    quality_df["Duplicate Content"] = (
        quality_df["Content Hash"].notna()
        & quality_df["Content Hash"].duplicated(keep=False)
    )

    display(
        quality_df[
            ["Filename", "Word Count", "Text Length",
             "Empty Text", "Duplicate Content"]
        ]
    )
else:
    print("No PDF or DOCX resumes found.")


,Filename,Word Count,Text Length,Empty Text,Duplicate Content
0,apurva cv new.docx,127,995,False,False
1,Nikita Salunke (3)-1.pdf,354,2613,False,False
2,Vaishnavi Dhumal AI Resume.docx,340,2753,False,False
3,Vaishnavi Dhumal DA Resume..pdf,289,2248,False,False
4,Vaishnavi Dhumal MIS Resume.docx,278,2264,False,False
5,webResume.pdf,227,1764,False,False


In [23]:
# Resume Feature Engineering and Outlier Checks

quality_df["Text Length"] = pd.to_numeric(
    quality_df["Text Length"], errors="coerce"
)
quality_df["Word Count"] = pd.to_numeric(
    quality_df["Word Count"], errors="coerce"
)

# Feature engineering
quality_df["Avg_Word_Length"] = (
    quality_df["Text Length"] /
    quality_df["Word Count"].replace(0, np.nan)
)

# Identify unusual word counts using IQR
Q1 = quality_df["Word Count"].quantile(0.25)
Q3 = quality_df["Word Count"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = max(0, Q1 - 1.5 * IQR)
upper_limit = Q3 + 1.5 * IQR

quality_df["Word_Count_Outlier"] = (
    (quality_df["Word Count"] < lower_limit) |
    (quality_df["Word Count"] > upper_limit)
)

quality_df["Data_Quality_Status"] = np.where(
    quality_df["Empty Text"],
    "Invalid - Empty Text",
    np.where(
        quality_df["Word_Count_Outlier"],
        "Review - Unusual Word Count",
        "OK"
    )
)

display(
    quality_df[
        ["Filename", "Word Count", "Text Length",
         "Word_Count_Outlier", "Data_Quality_Status"]
    ]
)

,Filename,Word Count,Text Length,Word_Count_Outlier,Data_Quality_Status
0,apurva cv new.docx,127,995,False,OK
1,Nikita Salunke (3)-1.pdf,354,2613,False,OK
2,Vaishnavi Dhumal AI Resume.docx,340,2753,False,OK
3,Vaishnavi Dhumal DA Resume..pdf,289,2248,False,OK
4,Vaishnavi Dhumal MIS Resume.docx,278,2264,False,OK
5,webResume.pdf,227,1764,False,OK
